# Project 10 — 3D Container Loading Optimizer
**Course:** Introduction to AI — Spring 2025-2026  
**Problem:** 3D Bin Packing Problem (3D-BPP)  

---

## Introduction

The **3D Bin Packing Problem (3D-BPP)** is a combinatorial optimization problem where the goal is to pack a set of 3D boxes into a container while maximizing space utilization. It is classified as **NP-hard**, meaning no known algorithm can solve it optimally in polynomial time for large inputs.

In this project, we tackle the problem using a standard **ISO 20-foot shipping container** (589 × 234 × 239 cm) and a dataset of 161 boxes across 12 types. We implement and compare three algorithms:

1. **Greedy — Best-Fit Decreasing (BFD)**
2. **Genetic Algorithm (GA)**
3. **Simulated Annealing (SA)**

All algorithms share the same placement decoder (`place_boxes`) which enforces:
- No overlapping boxes
- All boxes stay within container bounds
- Gravity constraint: every box must rest on the floor or on another box

---
## 1. Setup & Data Loading

In [1]:
import sys
import os
import time
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, os.path.abspath('..'))

from core.data_loader import load_data

DATA_PATH = '../data/project10_container_dataset.json'

boxes, container = load_data(DATA_PATH)

print(f'Container  : {container}')
print(f'Total boxes: {len(boxes)}')
print(f'Container volume: {container.volume():,} cm³')
print()

print('Sample boxes:')
for b in boxes[:5]:
    print(f'  {b}')

Container  : Container(589x234x239)
Total boxes: 386
Container volume: 32,940,414 cm³

Sample boxes:
  Box(id=0, 80x60x70, weight=35.0)
  Box(id=1, 80x60x70, weight=35.0)
  Box(id=2, 80x60x70, weight=35.0)
  Box(id=3, 80x60x70, weight=35.0)
  Box(id=4, 80x60x70, weight=35.0)


---
## 2. Algorithm 1 — Greedy Best-Fit Decreasing (BFD)

### How it works

The Greedy approach is the simplest and fastest algorithm:

1. **Sort** all boxes by volume in **descending order** (largest first).
2. For each box, try every available **Extreme Point** (candidate positions generated from corners of already-placed boxes).
3. Place the box at the **first valid position** that satisfies all constraints.

**Strength:** Very fast, good baseline.  
**Weakness:** Greedy decisions are irreversible — it can miss better arrangements.

In [2]:
from algorithms.greedy import solve as greedy_solve

start = time.time()
greedy_result = greedy_solve(boxes, container)
greedy_time = time.time() - start

greedy_placed = len(greedy_result['positions'])
greedy_util   = greedy_result['utilization']

print(f'Greedy Results')
print(f'  Boxes placed : {greedy_placed} / {len(boxes)}')
print(f'  Utilization  : {greedy_util:.2f}%')
print(f'  Time         : {greedy_time:.3f}s')

Greedy Results
  Boxes placed : 369 / 386
  Utilization  : 79.56%
  Time         : 10.691s


---
## 3. Algorithm 2 — Genetic Algorithm (GA)

### How it works

The Genetic Algorithm mimics natural evolution to search for better packings:

- **Chromosome:** A permutation of all 161 boxes, each paired with a rotation index (0–5). The order of boxes determines how they are packed.
- **Fitness:** `utilization% - penalty * unplaced_boxes + stability_bonus`
- **Selection:** Tournament selection — pick the best out of k random individuals.
- **Crossover:** Order Crossover (OX1) — preserves relative order of elements, required for permutation chromosomes.
- **Mutation:** Three operators — swap, inversion, and rotation mutation.
- **Elitism:** Top individuals always survive to the next generation.
- **Adaptive mutation:** Mutation rates increase during stagnation to escape local optima.

**Strength:** Explores a wide search space, finds good global solutions.  
**Weakness:** Slow — requires many fitness evaluations.

In [ ]:
from algorithms.genetic import solve as ga_solve

start = time.time()
ga_result = ga_solve(boxes, container)
ga_time = time.time() - start

ga_placed = len(ga_result['positions'])
ga_util   = ga_result['utilization']

print(f'Genetic Algorithm Results')
print(f'  Boxes placed : {ga_placed} / {len(boxes)}')
print(f'  Utilization  : {ga_util:.2f}%')
print(f'  Best fitness : {ga_result["fitness"]:.2f}')
print(f'  Best gen     : {ga_result["generation"]}')
print(f'  Time         : {ga_time:.2f}s')

---
## 4. Algorithm 3 — Simulated Annealing (SA)

### How it works

Simulated Annealing is inspired by the metallurgical process of slowly cooling metal:

- Start with an initial solution (e.g. random or greedy order).
- At each step, **swap two boxes** in the sequence to generate a neighbor.
- If the neighbor is **better**, always accept it.
- If the neighbor is **worse**, accept it with probability `e^(ΔE / T)` — this allows escaping local optima.
- **Temperature T** starts high (accepts bad moves freely) and cools down gradually (becomes more selective).

**Strength:** Can escape local optima, simpler than GA.  
**Weakness:** Sensitive to temperature schedule, slower than Greedy.

**Parameters used:**
- Initial temperature: 1000
- Cooling rate: 0.995

In [ ]:

#from algorithms.simulated_annealing import solve as sa_solve

#start = time.time()
#sa_result = sa_solve(boxes, container)
#sa_time = time.time() - start

#sa_placed = len(sa_result['positions'])
#sa_util   = sa_result['utilization']

#print(f'Simulated Annealing Results')
#print(f'  Boxes placed : {sa_placed} / {len(boxes)}')
#print(f'  Utilization  : {sa_util:.2f}%')
#print(f'  Time         : {sa_time:.2f}s')

---
## 5. Comparison

### Results Table

In [ ]:
print(f'{"Algorithm":<22} {"Boxes Placed":<15} {"Utilization":>12} {"Time (s)":>10}')
print('-' * 62)
print(f'{"Greedy (BFD)":<22} {str(greedy_placed)+" / "+str(len(boxes)):<15} {greedy_util:>11.2f}% {greedy_time:>10.3f}')
print(f'{"Genetic Algorithm":<22} {str(ga_placed)+" / "+str(len(boxes)):<15} {ga_util:>11.2f}% {ga_time:>10.2f}')
print(f'{"Simulated Annealing":<22} {"pending":<15} {"pending":>12} {"pending":>10}')

### Bar Chart — Utilization Comparison

In [ ]:
algorithms = ['Greedy', 'Genetic Algorithm']
utils      = [greedy_util, ga_util]
colors     = ['#3498DB', '#E74C3C']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(algorithms, utils, color=colors, width=0.45, zorder=3)

# Value labels on bars
for bar, val in zip(bars, utils):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.2f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=12)

# Theoretical max line
ax.axhline(y=34.14, color='gray', linestyle='--', linewidth=1.2, label='Theoretical max (34.14%)')

ax.set_ylabel('Volume Utilization (%)', fontsize=12)
ax.set_title('Algorithm Comparison — Volume Utilization', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.legend()

plt.tight_layout()
plt.savefig('../results/comparison_chart.png', dpi=150)
plt.show()
print('Chart saved to results/comparison_chart.png')

### Time Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(['Greedy', 'Genetic Algorithm'], [greedy_time, ga_time],
               color=['#3498DB', '#E74C3C'], height=0.4)

for bar, val in zip(bars, [greedy_time, ga_time]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}s', va='center', fontweight='bold')

ax.set_xlabel('Execution Time (seconds)', fontsize=12)
ax.set_title('Algorithm Speed Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Save Results to JSON

Results are saved so the frontend (Three.js visualization) can load them.

In [ ]:
def save_result(result, path):
    with open(path, 'w') as f:
        json.dump({
            'positions':   result['positions'],
            'utilization': result['utilization'],
            'container': {
                'L': container.L,
                'W': container.W,
                'H': container.H
            }
        }, f, indent=2)
    print(f'Saved: {path}')

save_result(greedy_result, '../results/solution_greedy.json')
save_result(ga_result,     '../results/solution_ga.json')
# save_result(sa_result,   '../results/solution_sa.json')  # uncomment when SA is ready

---
## 7. Conclusion

| Algorithm | Speed | Quality | Best For |
|---|---|---|---|
| Greedy |  Very Fast | Good baseline | Quick estimates |
| GA |  Slow | Best quality | When time allows |
| SA |  Medium | Good | Balance of speed & quality |

### Key Takeaways

- The **shared placement decoder** (`place_boxes`) ensures all algorithms are compared fairly under the same constraints.
- **Greedy** is fast but makes irreversible decisions — it misses better arrangements.
- **GA** explores a much wider search space through evolution, producing better utilization at the cost of time.
- **SA** offers a middle ground — it can escape local optima without the overhead of maintaining a full population.

### Future Improvements
- Enforce `orientation_locked` constraint for fragile boxes
- Enforce weight stacking (heavy boxes always below light ones)
- Allow box rotations in Greedy and SA
- Parallel fitness evaluation in GA for speedup